# Spotify Listening EDA

This notebook is used to understand listening patterns before applying machine learning.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

BASE_DIR = Path.cwd()
if BASE_DIR.name == 'notebooks':
    BASE_DIR = BASE_DIR.parent

processed = BASE_DIR / 'data' / 'processed'
figures = BASE_DIR / 'reports' / 'figures'
figures.mkdir(parents=True, exist_ok=True)


In [ ]:
df = pd.read_csv(processed / 'spotify_clean_history.csv')
track = pd.read_csv(processed / 'spotify_track_behavior_features.csv')
monthly = pd.read_csv(processed / 'spotify_monthly_behavior_features.csv')
user = pd.read_csv(processed / 'spotify_user_behavior_features.csv')

df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce', utc=True)
print('History:', df.shape)
print('Track features:', track.shape)
print('Monthly features:', monthly.shape)


## 1. Basic summary

In [ ]:
print('Total plays:', len(df))
print('Unique tracks:', df['master_metadata_track_name'].nunique())
print('Unique artists:', df['master_metadata_album_artist_name'].nunique())
print('Total minutes:', round(df['minutes_played'].sum(), 2))
print('Average minutes per play:', round(df['minutes_played'].mean(), 2))
print('Skip rate:', round(df['skipped'].fillna(False).astype(bool).mean() * 100, 2), '%')


## 2. Most played tracks

In [ ]:
top_tracks = (track.sort_values('total_plays', ascending=False)
             .head(10)[['master_metadata_track_name', 'master_metadata_album_artist_name', 'total_plays']])
print(top_tracks.to_string(index=False))

plt.figure(figsize=(9, 5))
plt.barh(top_tracks['master_metadata_track_name'][::-1], top_tracks['total_plays'][::-1])
plt.xlabel('Total plays')
plt.ylabel('Track')
plt.title('Top 10 Most Played Tracks')
plt.tight_layout()
plt.savefig(figures / 'top_tracks.png')
plt.show()


## 3. Most played artists

In [ ]:
top_artists = (df.groupby('master_metadata_album_artist_name')
              .size()
              .sort_values(ascending=False)
              .head(10))
print(top_artists)

plt.figure(figsize=(9, 5))
plt.barh(top_artists.index[::-1], top_artists.values[::-1])
plt.xlabel('Number of plays')
plt.ylabel('Artist')
plt.title('Top 10 Most Played Artists')
plt.tight_layout()
plt.savefig(figures / 'top_artists.png')
plt.show()


## 4. Listening by hour

In [ ]:
hour_counts = df['hour'].value_counts().sort_index()

plt.figure(figsize=(10, 4))
plt.plot(hour_counts.index, hour_counts.values, marker='o')
plt.xticks(range(24))
plt.xlabel('Hour of day')
plt.ylabel('Number of plays')
plt.title('Listening Activity by Hour')
plt.tight_layout()
plt.savefig(figures / 'listening_by_hour.png')
plt.show()


## 5. Monthly listening trend

In [ ]:
monthly['month'] = monthly['month'].astype(str)
monthly = monthly.sort_values('month')

plt.figure(figsize=(12, 4))
plt.plot(monthly['month'], monthly['total_plays'])
plt.xlabel('Month')
plt.ylabel('Total plays')
plt.title('Monthly Listening Trend')
plt.xticks(rotation=60)
plt.tight_layout()
plt.savefig(figures / 'monthly_listening_trend.png')
plt.show()


## 6. Skip behavior

In [ ]:
skip_rate = df['skipped'].fillna(False).astype(bool).mean()
print('Overall skip rate:', round(skip_rate * 100, 2), '%')

skip_by_hour = (df.groupby('hour')['skipped']
                .apply(lambda x: x.fillna(False).astype(bool).mean()))

plt.figure(figsize=(10, 4))
plt.plot(skip_by_hour.index, skip_by_hour.values * 100, marker='o')
plt.xticks(range(24))
plt.xlabel('Hour of day')
plt.ylabel('Skip rate (%)')
plt.title('Skip Rate by Hour')
plt.tight_layout()
plt.savefig(figures / 'skip_rate_by_hour.png')
plt.show()


## 7. Repeat vs discovery pattern

In [ ]:
repeat_rate = (track['total_plays'] > 1).mean()
discovery_rate = (track['total_plays'] == 1).mean()

print('Repeat rate:', round(repeat_rate * 100, 2), '%')
print('Discovery rate:', round(discovery_rate * 100, 2), '%')

labels = ['Repeated tracks', 'One-play tracks']
values = [(track['total_plays'] > 1).sum(), (track['total_plays'] == 1).sum()]

plt.figure(figsize=(6, 4))
plt.bar(labels, values)
plt.ylabel('Number of unique tracks')
plt.title('Repeated vs One-Play Tracks')
plt.tight_layout()
plt.savefig(figures / 'repeat_vs_discovery.png')
plt.show()


## 8. Weekend vs weekday

In [ ]:
weekend_rate = df['is_weekend'].mean()
weekday_rate = 1 - weekend_rate

print('Weekday listening:', round(weekday_rate * 100, 2), '%')
print('Weekend listening:', round(weekend_rate * 100, 2), '%')

plt.figure(figsize=(6, 4))
plt.bar(['Weekday', 'Weekend'], [weekday_rate * 100, weekend_rate * 100])
plt.ylabel('Share of plays (%)')
plt.title('Weekday vs Weekend Listening')
plt.tight_layout()
plt.savefig(figures / 'weekday_vs_weekend.png')
plt.show()


## 9. EDA observations

- Most played track/artist: BTS
- Highest listening hour: 10
- Highest listening month: October
- Overall skip rate: 26.11
- Repeat vs one-play pattern: 45-55
- Weekday vs weekend pattern: 70-30
